In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
 

In [ ]:
# Load a model
model = YOLO("models/yolo26n-seg.pt")  # load a pretrained model (recommended for training)
img1 = "images/000000204800.jpg"


In [ ]:
# Inference
results = model.predict(
    source=img1,
    conf=0.25,
    device="mps",   # Mac Apple Silicon；若不支援可改成 "cpu"
    verbose=False
)

# Plot segmentation result
annotated = results[0].plot()

In [ ]:
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

Class ID	Animal
14	bird
15	cat
16	dog
17	horse
18	sheep
19	cow
20	elephant
21	bear
22	zebra
23	giraffe

In [ ]:
animal_classes = list(range(14, 24))

results = model.predict(
    source=img1,
    conf=0.25,
    classes=animal_classes,
    device="mps",
    verbose=False
)

result = results[0]

annotated = result.plot(
    boxes=True,
    masks=True,
    labels=True
)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
annotated = result.plot(
    boxes=False,
    labels=False,
    masks=True
)

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
import numpy as np

result = results[0]

if result.masks is not None:
    masks = result.masks.data.cpu().numpy()
    classes = result.boxes.cls.cpu().numpy().astype(int)

    fig, axes = plt.subplots(
        1, len(masks),
        figsize=(5 * len(masks), 5),
        squeeze=False
    )

    for i, mask in enumerate(masks):
        axes[0, i].imshow(mask, cmap="gray")
        axes[0, i].set_title(
            f"Animal {i+1}: {model.names[classes[i]]}"
        )
        axes[0, i].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No animal detected.")

In [ ]:
import cv2
import matplotlib.pyplot as plt

result = results[0]
image = cv2.imread(img1)
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

dog_crops = []

for i, (box, cls) in enumerate(
    zip(result.boxes.xyxy.cpu().numpy(),
        result.boxes.cls.cpu().numpy())
):
    if model.names[int(cls)] != "dog":
        continue

    x1, y1, x2, y2 = map(int, box)

    crop = image_rgb[y1:y2, x1:x2]
    dog_crops.append({
        "image": crop,
        "offset": (x1, y1)
    })

    plt.figure(figsize=(5, 5))
    plt.imshow(crop)
    plt.title(f"Dog {len(dog_crops)}")
    plt.axis("off")
    plt.show()

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image

device = "mps" if torch.backends.mps.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(
    "IDEA-Research/grounding-dino-tiny"
)

dino = AutoModelForZeroShotObjectDetection.from_pretrained(
    "IDEA-Research/grounding-dino-tiny"
).to(device)

dino.eval()

In [ ]:
import matplotlib.patches as patches

for dog_idx, dog in enumerate(dog_crops):
    pil_image = Image.fromarray(dog["image"])

    inputs = processor(
        images=pil_image,
        text="eye.",
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = dino(**inputs)

    detections = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=0.30,
        text_threshold=0.30,
        target_sizes=[pil_image.size[::-1]]
    )[0]

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(pil_image)

    for box, score in zip(
        detections["boxes"],
        detections["scores"]
    ):
        x1, y1, x2, y2 = box.cpu().numpy()

        rect = patches.Rectangle(
            (x1, y1),
            x2-x1,
            y2-y1,
            linewidth=2,
            edgecolor="lime",
            facecolor="none"
        )
        ax.add_patch(rect)

        ax.text(
            x1, y1,
            f"Eye {score:.2f}",
            color="yellow"
        )

    ax.set_title(f"Dog {dog_idx+1}: Eye Detection")
    ax.axis("on")
    
    plt.show()

In [ ]:
from ultralytics import SAM
import numpy as np
import matplotlib.pyplot as plt

sam = SAM("models/sam2.1_t.pt")

# Grounding DINO 偵測的眼睛框
# 格式：[x1, y1, x2, y2]
eye_boxes = detections["boxes"].cpu().numpy()

# 用狗的 Crop 圖片做 Eye Segmentation
results_sam = sam.predict(
    source=dog_crops[1]["image"],
    bboxes=eye_boxes.tolist(),
    verbose=False
)

eye_masks = results_sam[0].masks.data.cpu().numpy()

plt.figure(figsize=(7, 5))
plt.imshow(dog_crops[1]["image"])

for mask in eye_masks:
    plt.imshow(
        np.ma.masked_where(mask == 0, mask),
        alpha=0.45,
        cmap="spring"
    )

ax.axis("on")
plt.show()

# Store 2 dog at one time

In [ ]:
import torch
import numpy as np
from PIL import Image

all_eye_detections = []

for dog_idx, dog in enumerate(dog_crops):

    pil_image = Image.fromarray(dog["image"])

    inputs = processor(
        images=pil_image,
        text="eye.",
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = dino(**inputs)

    detections = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=0.30,
        text_threshold=0.30,
        target_sizes=[pil_image.size[::-1]]
    )[0]

    # 儲存每隻狗的 detections
    all_eye_detections.append(detections)

    print(
        f"Dog {dog_idx+1}: "
        f"{len(detections['boxes'])} eye boxes detected"
    )

In [ ]:


all_eye_masks = []

for dog_idx, dog in enumerate(dog_crops):

    img = dog["image"]

    # 對應同一隻狗的 Grounding DINO boxes
    eye_boxes = (
        all_eye_detections[dog_idx]["boxes"]
        .cpu().numpy()
    )

    # 取 confidence 最高的兩個框
    scores = (
        all_eye_detections[dog_idx]["scores"]
        .cpu().numpy()
    )

    top2 = np.argsort(scores)[::-1][:2]
    eye_boxes = eye_boxes[top2]

    # SAM 2 inference
    result_sam = sam.predict(
        source=img,
        bboxes=eye_boxes.tolist(),
        verbose=False
    )[0]

    if result_sam.masks is None:
        print(f"Dog {dog_idx+1}: No eye mask")
        all_eye_masks.append([])
        continue

    masks = result_sam.masks.data.cpu().numpy()

    # 保存每隻狗的 Masks
    all_eye_masks.append(masks)

    # Visualization
    plt.figure(figsize=(7, 5))
    plt.imshow(img)

    for mask in masks:
        plt.imshow(
            np.ma.masked_where(mask == 0, mask),
            alpha=0.5,
            cmap="spring"
        )

    plt.title(f"Dog {dog_idx+1}: Eye Segmentation")
    plt.axis("off")
    plt.show()

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

H, W = image_rgb.shape[:2]

# 儲存所有動物的眼睛資料
all_eye_landmarks = []
all_eye_global_masks = []

for dog_idx, dog in enumerate(dog_crops):

    crop = dog["image"]
    x_offset, y_offset = dog["offset"]

    crop_h, crop_w = crop.shape[:2]

    masks = all_eye_masks[dog_idx]

    if len(masks) != 2:
        print(f"Dog {dog_idx+1}: Expected 2 eye masks")
        all_eye_landmarks.append(None)
        all_eye_global_masks.append(None)
        continue

    eye_data = []

    for mask in masks:

        # Resize mask to crop resolution if necessary
        if mask.shape != (crop_h, crop_w):
            mask = cv2.resize(
                mask.astype(np.float32),
                (crop_w, crop_h),
                interpolation=cv2.INTER_NEAREST
            )

        mask = mask > 0.5

        # Calculate local centroid
        ys, xs = np.where(mask)

        if len(xs) == 0:
            continue

        cx_local = xs.mean()
        cy_local = ys.mean()

        # Convert to original image coordinates
        cx_global = cx_local + x_offset
        cy_global = cy_local + y_offset

        # Restore mask to original image
        global_mask = np.zeros((H, W), dtype=np.uint8)

        global_mask[
            y_offset:y_offset+crop_h,
            x_offset:x_offset+crop_w
        ] = mask.astype(np.uint8)

        eye_data.append({
            "centroid": (cx_global, cy_global),
            "mask": global_mask
        })

    if len(eye_data) != 2:
        all_eye_landmarks.append(None)
        all_eye_global_masks.append(None)
        continue

    # Sort by X coordinate (front-facing dog)
    eye_data.sort(key=lambda e: e["centroid"][0])

    # Anatomical orientation
    right_eye = eye_data[0]
    left_eye = eye_data[1]

    all_eye_landmarks.append({
        "right_eye": right_eye["centroid"],
        "left_eye": left_eye["centroid"]
    })

    all_eye_global_masks.append({
        "right_eye": right_eye["mask"],
        "left_eye": left_eye["mask"]
    })

    print(f"Dog {dog_idx+1}")
    print(" Right eye:", right_eye["centroid"])
    print(" Left eye:", left_eye["centroid"])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(image_rgb)

colors = {
    "right_eye": "lime",
    "left_eye": "cyan"
}

for dog_idx, (landmarks, masks) in enumerate(
    zip(all_eye_landmarks, all_eye_global_masks)
):
    if landmarks is None:
        continue

    for eye_name in ["right_eye", "left_eye"]:

        mask = masks[eye_name]
        cx, cy = landmarks[eye_name]

        # Mask overlay
        ax.imshow(
            np.ma.masked_where(mask == 0, mask),
            cmap="spring",
            alpha=0.45
        )

        # Centroid
        ax.scatter(
            cx, cy,
            color=colors[eye_name],
            edgecolors="black",
            s=55,
            zorder=5
        )

        ax.text(
            cx + 5, cy - 8,
            f"Dog {dog_idx+1} {eye_name}",
            color=colors[eye_name],
            fontsize=9,
            weight="bold"
        )

ax.set_title("Eye Segmentation & Landmarks")
ax.axis("off")
plt.show()

In [ ]:
for i, landmarks in enumerate(all_eye_landmarks):
    print(f"Dog {i+1}")
    print("Right Eye:", landmarks["right_eye"])
    print("Left Eye:", landmarks["left_eye"])

In [ ]:
import numpy as np

def distance_between_points(p1, p2):
    return float(np.linalg.norm(np.array(p1) - np.array(p2)))

# SAM 2 predicted eye centroids
dog1 = all_eye_landmarks[0]
dog2 = all_eye_landmarks[1]

# 1. Dog 1 雙眼距離
pred_dog1 = distance_between_points(
    dog1["right_eye"],
    dog1["left_eye"]
)

# 2. Dog 2 雙眼距離
pred_dog2 = distance_between_points(
    dog2["right_eye"],
    dog2["left_eye"]
)

# 3. 兩隻狗的右眼距離
pred_between = distance_between_points(
    dog1["right_eye"],
    dog2["right_eye"]
)

print(f"Dog 1 Inter-eye Distance: {pred_dog1:.2f} px")
print(f"Dog 2 Inter-eye Distance: {pred_dog2:.2f} px")
print(f"Between Dogs Right-eye Distance: {pred_between:.2f} px")

In [ ]:
pip install pandas


#### from annotaion.ipynb

Dog 1 Inter-eye Distance: 40.73 px

Dog 2 Inter-eye Distance: 59.01 px

Between Dogs Right-eye Distance: 383.65 px

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Measurement": [
        "Dog 1 Inter-eye",
        "Dog 2 Inter-eye",
        "Between Dogs Right-eye"
    ],
    "GT (px)": [40.73, 59.01, 383.65],
    "Prediction (px)": [40.96, 58.33, 383.23]
})

df["Difference (px)"] = df["Prediction (px)"] - df["GT (px)"]
df["Absolute Error (px)"] = df["Difference (px)"].abs()
df["Error (%)"] = df["Absolute Error (px)"] / df["GT (px)"] * 100

display(df.round(2))

print(f"MAE: {df['Absolute Error (px)'].mean():.2f} px")
print(f"RMSE: {np.sqrt(np.mean(df['Difference (px)']**2)):.2f} px")
print(f"MAPE: {df['Error (%)'].mean():.2f}%")

In [ ]:
# from annotaion.ipynb
Dog 1 Inter-eye Distance: 40.73 px
Dog 2 Inter-eye Distance: 59.01 px
Between Dogs Right-eye Distance: 383.65 px

In [ ]:
from ultralytics import YOLO

model = YOLO("models/yolo26n-pose.pt")

In [ ]:


img1 = "images/000000204800.jpg"

results = model.predict(
    source=img1,
    conf=0.25,
    device="mps",
    verbose=False
)

result = results[0]

# Visualization
annotated = result.plot()

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
if result.keypoints is not None and len(result.boxes) > 0:
    keypoints = result.keypoints.xy.cpu().numpy()

    print("Keypoints shape:", keypoints.shape)
    print("Keypoints:", keypoints)
else:
    print("No pose detected.")
    

In [ ]:
%pip install ai-edge-litert huggingface_hub

In [ ]:
# dog-face-landmarks：https://huggingface.co/hugocornellier/dog-face-landmarks
from huggingface_hub import hf_hub_download

repo = "hugocornellier/dog-face-landmarks"

localizer_path = hf_hub_download(
    repo_id=repo,
    filename="dog_face_localizer.tflite"
)

landmark_path = hf_hub_download(
    repo_id=repo,
    filename="dog_face_landmarks_full.tflite"
)

print(localizer_path)
print(landmark_path)

In [ ]:
from ai_edge_litert.interpreter import Interpreter
import numpy as np
import cv2
import matplotlib.pyplot as plt

face_model = Interpreter(model_path=str(localizer_path))
face_model.allocate_tensors()

landmark_model = Interpreter(model_path=str(landmark_path))
landmark_model.allocate_tensors()

def run_tflite(model, image_tensor):
    input_info = model.get_input_details()[0]
    output_info = model.get_output_details()[0]

    model.set_tensor(
        input_info["index"],
        image_tensor.astype(np.float32)
    )
    model.invoke()

    return model.get_tensor(output_info["index"])

In [ ]:
def detect_dog_face(img):
    h, w = img.shape[:2]

    size = 224
    scale = min(size / w, size / h)

    new_w = round(w * scale)
    new_h = round(h * scale)

    left = (size - new_w) // 2
    top = (size - new_h) // 2

    resized = cv2.resize(img, (new_w, new_h))

    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    canvas[top:top+new_h, left:left+new_w] = resized

    input_tensor = canvas.astype(np.float32)[None] / 255.0

    output = run_tflite(face_model, input_tensor)
    box = np.asarray(output).reshape(-1)[:4]

    x1 = (box[0] * size - left) / scale
    y1 = (box[1] * size - top) / scale
    x2 = (box[2] * size - left) / scale
    y2 = (box[3] * size - top) / scale

    x1, x2 = sorted([x1, x2])
    y1, y2 = sorted([y1, y2])

    return np.array([
        np.clip(x1, 0, w),
        np.clip(y1, 0, h),
        np.clip(x2, 0, w),
        np.clip(y2, 0, h)
    ])

In [ ]:
dog_idx = 0
img = dog_crops[dog_idx]["image"]

face_box = detect_dog_face(img)
print("Face box:", face_box)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(img)

x1, y1, x2, y2 = face_box

ax.add_patch(plt.Rectangle(
    (x1, y1), x2-x1, y2-y1,
    fill=False, edgecolor="lime", linewidth=2
))

ax.set_title("Dog Face Localization")
ax.axis("off")
plt.show()


In [ ]:
def predict_dog_landmarks(img, face_box, margin=0.1):
    h, w = img.shape[:2]

    x1, y1, x2, y2 = map(float, face_box)

    bw = x2 - x1
    bh = y2 - y1

    x1 = max(0, int(np.floor(x1 - margin * bw)))
    y1 = max(0, int(np.floor(y1 - margin * bh)))
    x2 = min(w, int(np.ceil(x2 + margin * bw)))
    y2 = min(h, int(np.ceil(y2 + margin * bh)))

    if x2 <= x1 or y2 <= y1:
        raise ValueError("Invalid face box")

    face_crop = img[y1:y2, x1:x2]
    resized = cv2.resize(face_crop, (384, 384))

    input_tensor = resized.astype(np.float32)[None] / 255.0

    output = run_tflite(landmark_model, input_tensor)

    # 46 landmarks, each with normalized (x, y)
    points = np.asarray(output).reshape(46, 2)

    # Convert back to dog crop coordinates
    points[:, 0] = points[:, 0] * (x2-x1) + x1
    points[:, 1] = points[:, 1] * (y2-y1) + y1

    return points

In [ ]:
all_dog_landmarks = []

for dog_idx, dog in enumerate(dog_crops):
    img = dog["image"]

    face_box = detect_dog_face(img)
    points = predict_dog_landmarks(img, face_box)

    all_dog_landmarks.append(points)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(img)

    ax.scatter(
        points[:, 0],
        points[:, 1],
        s=15,
        c="lime"
    )

    for i, (x, y) in enumerate(points):
        ax.text(x+2, y-2, str(i), color="yellow", fontsize=7)

    ax.set_title(f"Dog {dog_idx+1} — 46 Facial Landmarks")
    ax.axis("off")
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

eye_groups = {
    "Eye A": [16, 18, 20, 22],
    "Eye B": [17, 19, 21, 23]
}

for dog_idx, dog in enumerate(dog_crops):
    img = dog["image"]
    points = all_dog_landmarks[dog_idx]

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(img)

    for name, indices in eye_groups.items():
        eye_pts = points[indices]
        center = eye_pts.mean(axis=0)

        ax.scatter(
            eye_pts[:, 0],
            eye_pts[:, 1],
            s=40,
            label=name
        )

        ax.scatter(
            center[0], center[1],
            marker="x",
            s=120,
            c="red"
        )

        for idx in indices:
            x, y = points[idx]
            ax.text(x + 3, y - 3, str(idx), color="yellow")

        print(f"Dog {dog_idx+1} {name}: {center}")

    ax.legend()
    ax.set_title(f"Dog {dog_idx+1} — Eye Landmarks")
    ax.axis("off")
    plt.show()

# Other Approach than using DINO ground

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = dog_crops[0]["image"]

gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

# 提升局部對比
clahe = cv2.createCLAHE(
    clipLimit=2.0,
    tileGridSize=(8, 8)
)
enhanced = clahe.apply(gray)

# 找深色區域
params = cv2.SimpleBlobDetector_Params()
params.minThreshold = 10
params.maxThreshold = 180
params.thresholdStep = 10

params.filterByColor = True
params.blobColor = 0

params.filterByArea = True
params.minArea = 8
params.maxArea = 500

params.filterByCircularity = True
params.minCircularity = 0.25

params.filterByConvexity = False
params.filterByInertia = False

detector = cv2.SimpleBlobDetector_create(params)
keypoints = detector.detect(enhanced)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(img)
ax[0].set_title("Original Dog Crop")

ax[1].imshow(img)

for kp in keypoints:
    x, y = kp.pt
    ax[1].scatter(x, y, c="lime", s=35)
    ax[1].add_patch(
        plt.Circle((x, y), kp.size / 2,
                   fill=False, color="yellow")
    )

ax[1].set_title(f"Dark Blob Candidates: {len(keypoints)}")

for a in ax:
    a.axis("off")

plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = dog_crops[0]["image"]
h, w = img.shape[:2]

# 先限制在上半部臉部區域
x1 = int(0.15 * w)
x2 = int(0.65 * w)
y1 = int(0.08 * h)
y2 = int(0.40 * h)

face_roi = img[y1:y2, x1:x2]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(img)
axes[0].add_patch(
    plt.Rectangle(
        (x1, y1), x2-x1, y2-y1,
        fill=False, edgecolor="lime", linewidth=2
    )
)
axes[0].set_title("Selected Face ROI")

axes[1].imshow(face_roi)
axes[1].set_title("Face ROI")

for ax in axes:
    ax.axis("off")

plt.show()

In [ ]:
gray = cv2.cvtColor(face_roi, cv2.COLOR_RGB2GRAY)

# 增強局部對比
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
gray = clahe.apply(gray)

# 以低亮度區域產生候選 Mask
threshold = np.percentile(gray, 20)
binary = (gray < threshold).astype(np.uint8) * 255

# 移除小雜訊
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (3, 3)
)

binary = cv2.morphologyEx(
    binary, cv2.MORPH_OPEN, kernel
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(face_roi)
axes[0].set_title("Face ROI")

axes[1].imshow(gray, cmap="gray")
axes[1].set_title("Grayscale + CLAHE")

axes[2].imshow(binary, cmap="gray")
axes[2].set_title("Dark Region Mask")

for ax in axes:
    ax.axis("off")

plt.show()

In [ ]:
num_labels, labels, stats, centroids = (
    cv2.connectedComponentsWithStats(
        binary, connectivity=8
    )
)

candidates = []

for i in range(1, num_labels):
    x, y, bw, bh, area = stats[i]

    if not (5 <= area <= 300):
        continue

    cx, cy = centroids[i]

    candidates.append({
        "center": (cx + x1, cy + y1),
        "area": area,
        "box": (x + x1, y + y1, bw, bh)
    })

fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(img)

for c in candidates:
    cx, cy = c["center"]
    ax.scatter(cx, cy, color="lime", s=35)

ax.set_title(f"Candidates after ROI + Threshold: {len(candidates)}")
ax.axis("off")
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = dog_crops[0]["image"]
h, w = img.shape[:2]

# Face ROI (先沿用你的設定)
x1, x2 = int(0.15*w), int(0.65*w)
y1, y2 = int(0.08*h), int(0.40*h)

face_roi = img[y1:y2, x1:x2]
gray = cv2.cvtColor(face_roi, cv2.COLOR_RGB2GRAY)

# Black-hat
kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (17, 17)
)

blackhat = cv2.morphologyEx(
    gray,
    cv2.MORPH_BLACKHAT,
    kernel
)

# Otsu threshold
_, binary = cv2.threshold(
    blackhat, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(face_roi)
axes[0].set_title("Face ROI")

axes[1].imshow(blackhat, cmap="gray")
axes[1].set_title("Black-hat Response")

axes[2].imshow(binary, cmap="gray")
axes[2].set_title("Candidate Dark Regions")

for ax in axes:
    ax.axis("off")

plt.show()